<a href="https://colab.research.google.com/github/Rokiafaysal/advanced-rag-chatbot/blob/main/rag_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf faiss-cpu sentence-transformers groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 16.0 MB/s eta 0:00:00


In [59]:
!pip install -U FlagEmbedding

In [18]:
!pip install transformers==4.44.0 FlagEmbedding --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 kB 11.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.8/177.8 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.1/147.1 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 83.1 MB/s eta 0:00:00


In [23]:
!pip install ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfu

In [24]:
!pip install langchain-groq langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 1.2.0
    Uninstalling groq-1.2.0:
      Successfully uninstalled groq-1.2.0


In [1]:
import pypdf
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq

In [2]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_35797/3120045098.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipykernel_35797/3120045098.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections i

In [3]:
from FlagEmbedding import FlagReranker

In [4]:
from google.colab import userdata
client = Groq(api_key=userdata.get('Groq_key'))


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
from pypdf import PdfReader
reader= PdfReader("/content/drive/MyDrive/1706.03762v7.pdf")

In [21]:
text = "\n".join(p.extract_text() for p in reader.pages)


In [22]:
model = SentenceTransformer('all-MiniLM-L6-v2')


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [23]:
def split_text(text, chunk_size=500,overlap=50):
  words = text.split()
  chunks=[]
  for i in range(0,len(words),chunk_size - overlap):
    chunk=" ".join(words[i:i+chunk_size])
    chunks.append(chunk)
  return chunks




In [24]:

chunks = split_text(text)
print(f" num of chunks: {len(chunks)}")

 num of chunks: 14


In [25]:
embedding=model.encode(chunks)
index=faiss.IndexFlatL2(embedding.shape[1])
index.add(embedding)
print({index.ntotal})

{14}


In [12]:
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

In [26]:
def search_reranked_verbose(query, k=10, top_n=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k)
    candidates = [chunks[i] for i in indices[0]]

    pairs = [[query, chunk] for chunk in candidates]
    scores = reranker.compute_score(pairs)

    ranked = sorted(zip(scores, candidates), reverse=True)

    for i, chunk in enumerate(candidates):
        print(f"{i+1}. {chunk[:100]}...")
        print("---")

    top = [chunk for _, chunk in ranked[:top_n]]
    for i, chunk in enumerate(top):
        print(f"{i+1}. {chunk[:100]}...")
        print("---")

    return top

results = search_reranked_verbose("what is deep learning")

1. Sennrich, Barry Haddow, and Alexandra Birch. Neural machine translation of rare words with subword u...
---
2. recognition. In Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition, pages...
---
3. entirely on self-attention to compute representations of its input and output without using sequence...
---
4. Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and...
---
5. that despite the lack of task-specific tuning our model performs sur- prisingly well, yielding bette...
---
6. the network. The shorter these paths between any combination of positions in the input and output se...
---
7. a compatibility function of the query with the corresponding key. 3.2.1 Scaled Dot-Product Attention...
---
8. V i ∈ Rdmodel×dv and W O ∈ Rhdv×dmodel. In this work we employ h = 8 parallel attention layers, or h...
---
9. of sequential operations for different layer types. n is the sequence length, d is the representatio...
---
1

In [14]:
model_chat="llama-3.3-70b-versatile"

In [27]:
def call_llm(query, chunks, model_chat=model_chat, temperature=0.4, max_tokens=300):
  context = "\n".join(chunks)
  prompt = f"""
You are an assistant helping students understand this book.

context from this book
{context}
    Question: {query}
   answer based only on the context above.

  """

  response = client.chat.completions.create(
    model=model_chat,
    messages=[
      {
        "role": "user",
        "content": prompt
      }
    ],
    temperature=temperature,
    max_tokens=max_tokens
  )

  return response.choices[0].message.content



In [28]:
results = search_reranked_verbose("what is deep learning")
answer = call_llm("what is deep learning", results)
print(answer)

1. Sennrich, Barry Haddow, and Alexandra Birch. Neural machine translation of rare words with subword u...
---
2. recognition. In Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition, pages...
---
3. entirely on self-attention to compute representations of its input and output without using sequence...
---
4. Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and...
---
5. that despite the lack of task-specific tuning our model performs sur- prisingly well, yielding bette...
---
6. the network. The shorter these paths between any combination of positions in the input and output se...
---
7. a compatibility function of the query with the corresponding key. 3.2.1 Scaled Dot-Product Attention...
---
8. V i ∈ Rdmodel×dv and W O ∈ Rhdv×dmodel. In this work we employ h = 8 parallel attention layers, or h...
---
9. of sequential operations for different layer types. n is the sequence length, d is the representatio...
---
1

In [29]:
from datasets import Dataset

questions = [
    {"question": "what is the attention mechanism",
     "reference": "Attention mechanism allows the model to focus on relevant parts of the input sequence"},
    {"question": "what is multi-head attention",
     "reference": "Multi-head attention runs attention multiple times in parallel and concatenates the results"},
    {"question": "what is the transformer architecture",
     "reference": "The transformer uses encoder and decoder stacks with self-attention and feed-forward layers"}
]

data = []
for q in questions:
    contexts = search_reranked_verbose(q["question"])
    answer = call_llm(q["question"], contexts)
    data.append({
        "question": q["question"],
        "answer": answer,
        "contexts": contexts,
        "reference": q["reference"]
    })

dataset = Dataset.from_list(data)

1. V i ∈ Rdmodel×dv and W O ∈ Rhdv×dmodel. In this work we employ h = 8 parallel attention layers, or h...
---
2. . <EOS> <pad> The Law will never be perfect , but its application should be just - this is what we a...
---
3. a compatibility function of the query with the corresponding key. 3.2.1 Scaled Dot-Product Attention...
---
4. entirely on self-attention to compute representations of its input and output without using sequence...
---
5. Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and...
---
6. the network. The shorter these paths between any combination of positions in the input and output se...
---
7. rows (A), we vary the number of attention heads and the attention key and value dimensions, keeping ...
---
8. of recurrent language models and encoder-decoder architectures [38, 24, 15]. Recurrent models typica...
---
9. recognition. In Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition, pages...
---
1

In [31]:
from ragas.llms import LangchainLLMWrapper
from langchain_groq import ChatGroq
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings

groq_llm = LangchainLLMWrapper(ChatGroq(
    api_key=userdata.get('Groq_key'),
    model="llama-3.1-8b-instant"
))

embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
))

result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=groq_llm,
    embeddings=embeddings
)

print(result)

/tmp/ipykernel_35797/2853943349.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  groq_llm = LangchainLLMWrapper(ChatGroq(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/tmp/ipykernel_35797/2853943349.py:11: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.em

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[6]: TimeoutError()


{'faithfulness': 1.0000, 'answer_relevancy': 0.8552, 'context_precision': 1.0000}
